# Preprocessing Dataset NHANES 2015-2016 untuk Prediksi Stroke

Notebook ini digunakan untuk mengunduh dataset NHANES 2015-2016, melakukan ekstraksi, penggabungan, dan preprocessing data untuk pemodelan prediksi stroke.

Fitur gaya hidup (tidur, stres, dan aktivitas fisik) disamakan masing-masing berjumlah **5 fitur** untuk menjaga keseimbangan kontribusi variabel gaya hidup.

## Alur Preprocessing
1. Install dan Import Library
2. Download dan Load Dataset NHANES
3. Penggabungan Dataset berdasarkan SEQN
4. Preprocessing Data:
   - Data Cleaning & Feature Encoding
   - Pengelompokan Fitur
   - Missing Value Handling
   - Feature Scaling & SMOTE
   - Simpan Output

## 1. Install dan Import Library

In [1]:
# Install library tambahan
!pip install imbalanced-learn xgboost shap joblib -q

import os
import joblib
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

print("Import library berhasil.")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Import library berhasil.


## 2. Download dan Load Dataset NHANES 2015-2016

In [2]:
# Mengunduh dataset dari repository Github jika belum ada
if not os.path.exists('NHANES-Stroke-Dataset'):
    os.system('git clone https://github.com/Adityamulyaf/NHANES-Stroke-Dataset.git')

# Menentukan path folder dataset
if os.path.exists('content/'):
    DATA_PATH = 'content/'
elif os.path.exists('NHANES-Stroke-Dataset/content/'):
    DATA_PATH = 'NHANES-Stroke-Dataset/content/'
else:
    DATA_PATH = '/content/NHANES-Stroke-Dataset/content/'

print("Path dataset yang digunakan:", DATA_PATH)
print("Daftar file dataset:")
for file in sorted(os.listdir(DATA_PATH)):
    print(f" - {file}")

Path dataset yang digunakan: content/
Daftar file dataset:
 - BMX_I.xpt
 - BPQ_I.xpt
 - BPX_I.xpt
 - DEMO_I.xpt
 - DIQ_I.xpt
 - DPQ_I.xpt
 - MCQ_I.xpt
 - PAQ_I.xpt
 - SLQ_I.xpt
 - SMQ_I.xpt


In [3]:
# Memuat file SAS XPORT (.xpt)
dfs = {}
files = {
    'demo': 'DEMO_I.xpt',
    'mcq':  'MCQ_I.xpt',
    'bpq':  'BPQ_I.xpt',
    'diq':  'DIQ_I.xpt',
    'bmx':  'BMX_I.xpt',
    'bpx':  'BPX_I.xpt',
    'smq':  'SMQ_I.xpt',
    'slq':  'SLQ_I.xpt',
    'dpq':  'DPQ_I.xpt',
    'paq':  'PAQ_I.xpt',
}

for name, filename in files.items():
    path = os.path.join(DATA_PATH, filename)
    print(f"Loading {filename}...")
    try:
        dfs[name] = pd.read_sas(path, format='xport', encoding='utf-8')
        print(f"  Berhasil: {dfs[name].shape[0]} baris, {dfs[name].shape[1]} kolom")
    except Exception as e:
        print(f"  Gagal memuat {filename}: {e}")

Loading DEMO_I.xpt...
  Berhasil: 9971 baris, 47 kolom
Loading MCQ_I.xpt...
  Berhasil: 9575 baris, 90 kolom
Loading BPQ_I.xpt...
  Berhasil: 6327 baris, 11 kolom
Loading DIQ_I.xpt...
  Berhasil: 9575 baris, 54 kolom
Loading BMX_I.xpt...


  Berhasil: 9544 baris, 26 kolom


Loading BPX_I.xpt...


  Berhasil: 9544 baris, 21 kolom
Loading SMQ_I.xpt...
  Berhasil: 7001 baris, 42 kolom
Loading SLQ_I.xpt...
  Berhasil: 6327 baris, 8 kolom
Loading DPQ_I.xpt...
  Berhasil: 5735 baris, 11 kolom
Loading PAQ_I.xpt...
  Berhasil: 9255 baris, 94 kolom


## 3. Seleksi Kolom dan Penggabungan Dataset

In [4]:
# ── Demografi ────────────────────────────────────────────────────────────────
demo_cols = ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'DMDEDUC2', 'INDFMPIR']
demo_clean = dfs['demo'][demo_cols].copy()

# ── Target (Stroke) & Riwayat Jantung ────────────────────────────────────────
mcq_cols = ['SEQN', 'MCQ160F', 'MCQ160B', 'MCQ160C', 'MCQ160E']
mcq_clean = dfs['mcq'][mcq_cols].copy()

# ── Hipertensi & Diabetes ─────────────────────────────────────────────────────
bpq_cols = ['SEQN', 'BPQ020']
bpq_clean = dfs['bpq'][bpq_cols].copy()

diq_cols = ['SEQN', 'DIQ010']
diq_clean = dfs['diq'][diq_cols].copy()

# ── Pemeriksaan Fisik ─────────────────────────────────────────────────────────
bmx_cols = ['SEQN', 'BMXBMI', 'BMXWAIST']
bmx_clean = dfs['bmx'][bmx_cols].copy()

bpx_cols = ['SEQN', 'BPXSY1', 'BPXDI1']
bpx_clean = dfs['bpx'][bpx_cols].copy()

# ── Kebiasaan Merokok ─────────────────────────────────────────────────────────
smq_cols = ['SEQN', 'SMQ020', 'SMQ040']
smq_clean = dfs['smq'][smq_cols].copy()

# ── Fitur Tidur (5 Fitur) ─────────────────────────────────────────────────────
# SLD012: jam tidur | SLQ030: frekuensi mendengkur | SLQ040: sleep apnea
# SLQ050: masalah tidur ke dokter | SLQ120: kantuk siang
slq_cols = ['SEQN', 'SLD012', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ120']
slq_clean = dfs['slq'][slq_cols].copy()

# ── Fitur Stres (5 Fitur) ───────────────────────────────
# DPQ010: anhedonia | DPQ020: depresi | DPQ040: kelelahan
# DPQ050: konsentrasi | DPQ060: harga diri rendah
dpq_selected = ['DPQ010', 'DPQ020', 'DPQ040', 'DPQ050', 'DPQ060']
dpq_cols = ['SEQN'] + dpq_selected
dpq_clean = dfs['dpq'][dpq_cols].copy()

# Bersihkan nilai tidak valid (hanya 0-3 yang valid untuk PHQ)
for col in dpq_selected:
    dpq_clean[col] = dpq_clean[col].apply(lambda x: x if pd.notna(x) and x <= 3.0 else np.nan)

# Rename ke nama deskriptif
dpq_rename = {
    'DPQ010': 'stress_anhedonia',
    'DPQ020': 'stress_depressed',
    'DPQ040': 'stress_fatigue',
    'DPQ050': 'stress_concentration',
    'DPQ060': 'stress_self_esteem'
}
dpq_clean = dpq_clean.rename(columns=dpq_rename)
stress_cols_keep = ['SEQN', 'stress_anhedonia', 'stress_depressed',
                    'stress_fatigue', 'stress_concentration', 'stress_self_esteem']
dpq_clean = dpq_clean[stress_cols_keep]

# ── Fitur Aktivitas Fisik (5 Fitur) ──────────────────────────────────────────
# PAQ605: kerja fisik berat | PAQ650: olahraga berat | PAD660: menit olahraga berat
# PAQ665: olahraga sedang | PAD680: menit sedentary
paq_cols = ['SEQN', 'PAQ605', 'PAQ650', 'PAD660', 'PAQ665', 'PAD680']
paq_clean = dfs['paq'][paq_cols].copy()

# ── Gabungkan semua dataset berdasarkan SEQN ─────────────────────────────────
df = demo_clean
for other in [mcq_clean, bpq_clean, diq_clean, bmx_clean,
              bpx_clean, smq_clean, slq_clean, dpq_clean, paq_clean]:
    df = df.merge(other, on='SEQN', how='left')

print(f"Total baris setelah penggabungan : {df.shape[0]}")
print(f"Total kolom setelah penggabungan : {df.shape[1]}")

Total baris setelah penggabungan : 9971
Total kolom setelah penggabungan : 33


## 4. Preprocessing Data

### A. Data Cleaning & Feature Encoding

In [5]:
# ── Target: Stroke ────────────────────────────────────────────────────────────
# MCQ160F: 1 = pernah stroke, 2 = tidak pernah
df['stroke'] = df['MCQ160F'].map({1.0: 1, 2.0: 0})
df = df[df['stroke'].notna()].copy()

print("Distribusi variabel stroke:")
print(df['stroke'].value_counts())
print(f"Rasio stroke positif: {df['stroke'].mean():.2%}")

# Filter usia dewasa (≥18 tahun)
df = df[df['RIDAGEYR'] >= 18].copy()
print(f"\nJumlah data setelah filter usia dewasa: {len(df)} baris")

Distribusi variabel stroke:
stroke
0.0    5505
1.0     209
Name: count, dtype: int64
Rasio stroke positif: 3.66%

Jumlah data setelah filter usia dewasa: 5714 baris


In [6]:
# ── Encoding variabel biner ───────────────────────────────────────────────────
binary_cols = {
    'BPQ020':  'hypertension',         # 1=Ya, 2=Tidak
    'DIQ010':  'diabetes',              # 1=Ya, 2=Tidak, 3=Borderline→1
    'MCQ160B': 'heart_failure',
    'MCQ160C': 'coronary_disease',
    'MCQ160E': 'heart_attack',
    'SMQ020':  'ever_smoked',
    'SLQ050':  'sleep_problem_doctor',
    'PAQ605':  'vigorous_work',
    'PAQ650':  'vigorous_leisure',
    'PAQ665':  'moderate_leisure',
}
for old, new in binary_cols.items():
    if old in df.columns:
        df[new] = df[old].map({1.0: 1, 2.0: 0, 3.0: 1})

# SMQ040: 1=setiap hari, 2=kadang-kadang → masih merokok (1)
#          3=tidak sama sekali            → tidak merokok (0)
df['current_smoker'] = df['SMQ040'].map({1.0: 1, 2.0: 1, 3.0: 0})

# ── Rename kolom ─────────────────────────────────────────────────────────────
rename_map = {
    'RIDAGEYR': 'age',
    'RIAGENDR': 'gender',
    'RIDRETH3': 'race',
    'DMDEDUC2': 'education',
    'INDFMPIR': 'income_ratio',
    'BMXBMI':   'bmi',
    'BMXWAIST': 'waist_circ',
    'BPXSY1':   'systolic_bp',
    'BPXDI1':   'diastolic_bp',
    'SLD012':   'sleep_hours',
    'SLQ030':   'snoring_freq',
    'SLQ040':   'sleep_apnea',
    'SLQ120':   'daytime_sleepy',
    'PAD660':   'vigorous_leisure_min',
    'PAD680':   'sedentary_min',
}
df = df.rename(columns=rename_map)

# gender: 1=Laki-laki -> 1, 2=Perempuan -> 0
df['gender'] = df['gender'].map({1.0: 1, 2.0: 0})

print("Encoding dan pemetaan fitur selesai.")

Encoding dan pemetaan fitur selesai.


### B. Pengelompokan Fitur

Masing-masing kelompok gaya hidup (tidur, stres, aktivitas fisik) terdiri dari **5 fitur** untuk menjaga kesetaraan dalam analisis SHAP group contribution.

**Perubahan dari versi sebelumnya:**
- `stress_score` dihapus (redundan/multikolinear dengan item individual) → diganti `stress_concentration` (DPQ050)
- `vigorous_work_min` (PAD615, MNAR ~89%) diganti `vigorous_leisure_min` (PAD660) yang missing-nya lebih acak
- `current_smoker` di-encode biner: 1=masih merokok, 0=tidak merokok

In [7]:
# ── Definisi kelompok fitur ───────────────────────────────────────────────────
CLINICAL = [
    'age', 'gender', 'race', 'education', 'income_ratio',
    'bmi', 'waist_circ', 'systolic_bp', 'diastolic_bp',
    'hypertension', 'diabetes', 'heart_failure',
    'coronary_disease', 'heart_attack',
    'ever_smoked', 'current_smoker'
]

SLEEP = [
    'sleep_hours',          # jam tidur per malam
    'snoring_freq',         # frekuensi mendengkur
    'sleep_apnea',          # apnea saat tidur
    'sleep_problem_doctor', # pernah konsultasi dokter soal tidur
    'daytime_sleepy'        # kantuk di siang hari
]

# FIX: 5 item PHQ individual, stress_score dihapus
STRESS = [
    'stress_anhedonia',     # DPQ010: kehilangan minat
    'stress_depressed',     # DPQ020: merasa tertekan
    'stress_fatigue',       # DPQ040: kelelahan
    'stress_concentration', # DPQ050: kesulitan berkonsentrasi (BARU)
    'stress_self_esteem'    # DPQ060: harga diri rendah
]

# FIX: vigorous_work_min → vigorous_leisure_min
PHYSICAL = [
    'vigorous_work',         # PAQ605: melakukan kerja fisik berat (biner)
    'vigorous_leisure',      # PAQ650: olahraga berat di waktu luang (biner)
    'vigorous_leisure_min',  # PAD660: menit olahraga berat per minggu (BARU)
    'moderate_leisure',      # PAQ665: olahraga sedang di waktu luang (biner)
    'sedentary_min'          # PAD680: menit sedentary per hari
]

TARGET = 'stroke'

ALL_COLS = CLINICAL + SLEEP + STRESS + PHYSICAL + [TARGET]
df_model = df[ALL_COLS].copy()

print(f"Total fitur yang digunakan : {len(ALL_COLS) - 1}")
print(f" - Fitur Klinis            : {len(CLINICAL)}")
print(f" - Fitur Tidur             : {len(SLEEP)}")
print(f" - Fitur Stres             : {len(STRESS)}")
print(f" - Fitur Aktivitas Fisik   : {len(PHYSICAL)}")

Total fitur yang digunakan : 31
 - Fitur Klinis            : 16
 - Fitur Tidur             : 5
 - Fitur Stres             : 5
 - Fitur Aktivitas Fisik   : 5


### C. Missing Value Handling

Strategi:
1. Drop baris yang missing di fitur klinis inti
2. **FIX `vigorous_leisure_min`**: jika responden tidak melakukan vigorous leisure (`vigorous_leisure == 0`), maka menit-nya diisi 0 (bukan NaN) - karena missing bukan karena data hilang, melainkan karena tidak relevan
3. Imputasi **modus** untuk fitur ordinal/biner
4. Imputasi **median** untuk fitur numerik kontinyu

In [8]:
print("Missing values sebelum penanganan:")
missing = df_model.isnull().sum()
print(missing[missing > 0].to_string())

# ── 1. Drop baris missing di fitur klinis inti ────────────────────────────────
core_cols = ['age', 'bmi', 'systolic_bp', 'hypertension', 'diabetes', TARGET]
df_model = df_model.dropna(subset=core_cols)

# ── 2. FIX: vigorous_leisure_min - isi 0 jika tidak melakukan vigorous leisure ─
# PAD660 hanya diisi jika PAQ650=1 (Ya). Jika PAQ650=0 → menit = 0, bukan NaN.
# Ini bukan missing at random - ini MNAR yang bisa diselesaikan secara logis.
mask_no_vigorous = df_model['vigorous_leisure'] == 0
df_model.loc[mask_no_vigorous, 'vigorous_leisure_min'] = \
    df_model.loc[mask_no_vigorous, 'vigorous_leisure_min'].fillna(0)

print(f"\nMissing vigorous_leisure_min setelah perbaikan logis: "
      f"{df_model['vigorous_leisure_min'].isnull().sum()}")

# ── 3. Imputasi modus untuk kolom ordinal/biner ───────────────────────────────
ordinal_cols = ['snoring_freq', 'sleep_apnea', 'daytime_sleepy',
                'current_smoker', 'ever_smoked']
for col in ordinal_cols:
    if col in df_model.columns and df_model[col].isnull().sum() > 0:
        modus = df_model[col].mode()[0]
        df_model[col] = df_model[col].fillna(modus)

# ── 4. Imputasi median untuk fitur numerik kontinyu sisanya ───────────────────
num_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]
imputer = SimpleImputer(strategy='median')
df_model[num_cols] = imputer.fit_transform(df_model[num_cols])

print(f"\nJumlah baris setelah pembersihan  : {len(df_model)}")
print(f"Total missing value tersisa       : {df_model.isnull().sum().sum()}")

Missing values sebelum penanganan:
income_ratio             635
bmi                      312
waist_circ               600
systolic_bp              566
diastolic_bp             566
hypertension               6
diabetes                   3
heart_failure             11
coronary_disease          25
heart_attack               8
ever_smoked               10
current_smoker          3330
sleep_hours               32
sleep_problem_doctor       4
stress_anhedonia         813
stress_depressed         803
stress_fatigue           803
stress_concentration     806
stress_self_esteem       805
vigorous_work              2
vigorous_leisure           1
vigorous_leisure_min    4304
moderate_leisure           4
sedentary_min             11

Missing vigorous_leisure_min setelah perbaikan logis: 5

Jumlah baris setelah pembersihan  : 5090
Total missing value tersisa       : 0


### D. Pembagian Data, Feature Scaling, dan SMOTE

In [9]:
# ── Pisahkan fitur dan target ─────────────────────────────────────────────────
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET].astype(int)

# ── Split 80:20, stratified ───────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── StandardScaler: fit pada X_train, transform X_train dan X_test ────────────
scaler = StandardScaler()
X_train_scaled_raw = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Distribusi kelas sebelum SMOTE (Training):")
print(y_train.value_counts().to_dict())

# ── SMOTE dilakukan pada data yang sudah discaling ────────────────────────────
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_scaled, y_train_resampled = smote.fit_resample(X_train_scaled_raw, y_train)

print("Distribusi kelas setelah SMOTE (Training):")
print(pd.Series(y_train_resampled).value_counts().to_dict())

print(f"\nBentuk X_train (setelah scaling + SMOTE) : {X_train_scaled.shape}")
print(f"Bentuk X_test  (data asli + scaling)      : {X_test_scaled.shape}")
print("\nPreprocessing selesai.")

Distribusi kelas sebelum SMOTE (Training):
{0: 3930, 1: 142}


Distribusi kelas setelah SMOTE (Training):
{0: 3930, 1: 3930}

Bentuk X_train (setelah scaling + SMOTE) : (7860, 31)
Bentuk X_test  (data asli + scaling)      : (1018, 31)

Preprocessing selesai.


### E. Simpan Output untuk Notebook Training

In [10]:
# Simpan df_model bersih (sebelum split/SMOTE/scale)
# supaya notebook training bisa subset fitur per skenario eksperimen
df_model.to_csv('nhanes_clean.csv', index=False)
print("nhanes_clean.csv tersimpan.")

# Simpan definisi kelompok fitur
joblib.dump(
    {'CLINICAL': CLINICAL, 'SLEEP': SLEEP,
     'STRESS': STRESS, 'PHYSICAL': PHYSICAL},
    'feature_groups.pkl'
)
print("feature_groups.pkl tersimpan.")

# Simpan data yang sudah di-resample (SMOTE) dan di-scale ke CSV
# Latih (Training)
df_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
df_train_scaled['stroke'] = y_train_resampled.reset_index(drop=True) if hasattr(y_train_resampled, 'reset_index') else y_train_resampled
df_train_scaled.to_csv('nhanes_train_scaled.csv', index=False)
print("nhanes_train_scaled.csv tersimpan.")

# Uji (Testing)
df_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)
df_test_scaled['stroke'] = y_test.reset_index(drop=True)
df_test_scaled.to_csv('nhanes_test_scaled.csv', index=False)
print("nhanes_test_scaled.csv tersimpan.")

# Download dari Colab jika tersedia
try:
    from google.colab import files
    files.download('nhanes_clean.csv')
    files.download('feature_groups.pkl')
    files.download('nhanes_train_scaled.csv')
    files.download('nhanes_test_scaled.csv')
    print("File berhasil didownload.")
except:
    print("(Tidak di Colab - file tersimpan di direktori lokal)")

nhanes_clean.csv tersimpan.
feature_groups.pkl tersimpan.


nhanes_train_scaled.csv tersimpan.
nhanes_test_scaled.csv tersimpan.
(Tidak di Colab - file tersimpan di direktori lokal)
